In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 04 - Results & Reporting\n",
    "\n",
    "Forecasting, visualization, and comprehensive report generation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "sys.path.insert(0, os.path.join(os.getcwd(), '..'))\n",
    "\n",
    "from src.data_generator import ESGDataGenerator\n",
    "from src.data_processor import DataProcessor\n",
    "from src.arima_model import ARIMAForecaster\n",
    "from src.diagnostics import ModelDiagnostics\n",
    "from src.visualization import Visualizer\n",
    "from src.reporting import ReportGenerator\n",
    "\n",
    "sns.set_style('whitegrid')\n",
    "plt.rcParams['figure.figsize'] = (14, 8)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Complete Pipeline"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate and process\n",
    "generator = ESGDataGenerator(n_periods=36, missing_rate=0.15, seed=42)\n",
    "df_original = generator.generate()\n",
    "\n",
    "processor = DataProcessor(df_original)\n",
    "df_processed = processor.handle_missing_values(method='cubic')\n",
    "\n",
    "# Model\n",
    "forecaster = ARIMAForecaster(df_processed['Environmental'])\n",
    "forecaster.auto_fit(verbose=False)\n",
    "forecaster.fit_arima(verbose=False)\n",
    "\n",
    "# Diagnostics\n",
    "diagnostics = ModelDiagnostics(forecaster.results, df_processed['Environmental'])\n",
    "diagnostics.run_all_diagnostics(verbose=False)\n",
    "\n",
    "print(\"✓ Pipeline complete!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Forecasting"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate forecast\n",
    "forecast_result = forecaster.forecast(steps=12, confidence=0.95)\n",
    "\n",
    "# Display\n",
    "print(\"\\nForecast Results:\")\n",
    "print(forecast_result['forecast_df'].head(10))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Visualizations"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create visualizer\n",
    "visualizer = Visualizer()\n",
    "\n",
    "# Forecast plot\n",
    "forecast_dates = pd.date_range(\n",
    "    start=df_processed['Date'].iloc[-1] + pd.DateOffset(months=1),\n",
    "    periods=12, freq='M'\n",
    ")\n",
    "\n",
    "visualizer.plot_arima_forecast(\n",
    "    df_processed['Environmental'],\n",
    "    forecast_result['forecast'].values,\n",
    "    forecast_dates,\n",
    "    (forecast_result['confidence_intervals'].iloc[:, 0].values,\n",
    "     forecast_result['confidence_intervals'].iloc[:, 1].values),\n",
    "    'Environmental Score'\n",
    ")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Residual Diagnostics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot residuals\n",
    "visualizer.plot_residuals_diagnostics(forecaster.results.resid)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Comprehensive Report"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate report\n",
    "reporter = ReportGenerator(\n",
    "    forecaster,\n",
    "    df_original,\n",
    "    df_processed,\n",
    "    diagnostics\n",
    ")\n",
    "\n",
    "report = reporter.generate_full_report(save_files=False)\n",
    "reporter.print_report_summary(report)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Export Results"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Export forecast\n",
    "forecast_df = forecast_result['forecast_df'].copy()\n",
    "print(\"\\nForecast DataFrame:\")\n",
    "print(forecast_df)\n",
    "\n",
    "# Export data summary\n",
    "data_summary = processor.get_summary_statistics()\n",
    "print(\"\\nData Summary:\")\n",
    "print(data_summary)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}
